In [1]:
## load packages 
import pandas as pd
import re
import numpy as np
# import plotnine
# from plotnine import *
import pickle

## nltk imports
from nltk.tokenize import word_tokenize, wordpunct_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

## sklearn imports
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

## print mult things
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## random
import random

pd.set_option('display.max_colwidth', None)

In [2]:
tcm_df = pd.read_csv("../Data/cleanedData.csv")
tcm_df.head()

,content_type,creation_time,hashtags,id,is_branded_content,lang,match_type,mcl_url,modified_time,multimedia,...,statistics.comment_count,statistics.like_count,statistics.views,statistics.views_date_last_refreshed,text,hashtags_extracted,text_clean,week,first_week,is_new
0,videos,2026-05-13 11:57:53,"[""talhaahadroundtable"",""talhaahadpodcast"",""talhaahad"",""aiseries"",""tcm""]",1656727755658052,False,en,"[""post_text""]",https://www.facebook.com/transparency-tools/content-library/dataset/1119037145491882/instagram/post/1656727755658052/,2026-05-13T13:13:23+00:00,"[{""id"":""1656727755658052"",""type"":""video"",""duration"":96.98,""tags"":{""0"":""815162807738463"",""1"":""363842936492074"",""2"":""1605108547564743"",""3"":""862811636124438""},""url"":""https://lookaside.facebook.com/mcl/multimedia/lookaside/?download_info=AZs-aB7bOGKbGjI-CcOOv4JNNXZrbNDrHSoeDEePp-cNP-QBiwXN0zt7na3Gm3UOJcS3xz0cxnZzMX8-t-DSIP3E0oVnPB_FEZTvYs01NDdf6bjNJc3IQyLO-kqRy4KnFzkuYs69f0sYV3EsVfmB""}]",...,0.0,7.0,644.0,2026-05-13,"Ask AI a wrong question, it'll give you a confident wrong answer.\nHow do you catch it?\nOnly if your fundamentals are solid.\n\nWatch the episode with @atom.camp on my youtube channel:\nThe link is in my Bio.\n\n#talhaahad #talhaahadroundtable #aiseries #tcm #talhaahadpodcast","['#talhaahad', '#talhaahadroundtable', '#aiseries', '#tcm', '#talhaahadpodcast']","Ask AI a wrong question, it'll give you a confident wrong answer. How do you catch it? Only if your fundamentals are solid. Watch the episode with @atom.camp on my youtube channel: The link is in my Bio.",2026-05-11,2025-07-21,False
1,videos,2026-05-13 10:49:04,"[""titancapitalmarkets"",""tcm""]",898235766608248,False,en,"[""post_text""]",https://www.facebook.com/transparency-tools/content-library/dataset/1119037145491882/instagram/post/898235766608248/,2026-05-13T11:39:08+00:00,"[{""id"":""898235766608248"",""type"":""video"",""duration"":42.26,""tags"":{""0"":""2541861929546394""},""url"":""https://lookaside.facebook.com/mcl/multimedia/lookaside/?download_info=AZuPZBr281UWDYmMxBFdpF7q_HRsA7IHXJayOwult3szEQfL87-Tj6PeXpAveoOKLNL-CC5nYmIq46JkqheKVGD7rAZ9-XTBvX3hbNROLYptzADWElKWRlpUO94g524jJhPa3qsKROJuBuA6Ue4""}]",...,3.0,37.0,1339.0,2026-05-13,"From making hits…\nto hitting payouts.\n\n5ivebeatz took the same discipline, consistency, and focus from music and applied it to trading.\n\nNow it’s paying at scale.\n\n#titancapitalmarkets #tcm","['#titancapitalmarkets', '#tcm']","From making hits… to hitting payouts. 5ivebeatz took the same discipline, consistency, and focus from music and applied it to trading. Now it’s paying at scale.",2026-05-11,2025-05-12,False
2,videos,2026-05-13 02:11:45,"[""healthy"",""missmaya"",""tcm"",""wellness"",""guasha""]",980906014454966,False,en,"[""post_text""]",https://www.facebook.com/transparency-tools/content-library/dataset/1119037145491882/instagram/post/980906014454966/,2026-05-13T02:42:14+00:00,"[{""id"":""980906014454966"",""type"":""video"",""duration"":45.3,""url"":""https://lookaside.facebook.com/mcl/multimedia/lookaside/?download_info=AZtbkpMMe80Yo02bIqwpLH_xfjNewRnxw3W-S-04QtJFjR_CTyv7EMiLD0eWDy71hZOqRIlAJxw-clZEadlcjfOqmFs852OTRG25zEOj-_I1MrriR0oXe-33Da0WkeUZOyARoveGh0DTVl0l_48""}]",...,0.0,3.0,861.0,2026-05-13,#tcm #missmaya #guasha #healthy #wellness,"['#tcm', '#missmaya', '#guasha', '#healthy', '#wellness']",NaN,2026-05-11,2025-09-01,False
3,albums,2026-05-13 01:57:36,"[""chinesemedicineworks"",""pmos"",""tcm"",""acupuncture"",""pcos""]",1016501597388882,False,en,"[""post_text""]",https://www.facebook.com/transparency-tools/content-library/dataset/1119037145491882/instagram/post/1016501597388882/,2026-05-13T01:59:21+00:00,"[{""id"":""744512645356645"",""type"":""photo"",""url"":""https://lookaside.facebook.com/mcl/multimedia/lookaside/?download_info=AZuTYeNjVDiGNxQ5mKY-ZdtBJF_TINkNTuq5cU0AU2KwTcoRukWudD7WmrkG0mTt8irenEitKfvF44FaOL9elWGPneOiQM_8FmCuDN6FwldYujkKlf-sd3LHs-_KXOmZRCP5M3naf4ntE27GPOQ""},{""id"":""2141712803273281

In [4]:
# selecting only creator accounts 
tcm_df_creator = tcm_df[tcm_df["post_owner.type"] == "creator"]
tcm_df.head()

,content_type,creation_time,hashtags,id,is_branded_content,lang,match_type,mcl_url,modified_time,multimedia,...,statistics.comment_count,statistics.like_count,statistics.views,statistics.views_date_last_refreshed,text,hashtags_extracted,text_clean,week,first_week,is_new
0,videos,2026-05-13 11:57:53,"[""talhaahadroundtable"",""talhaahadpodcast"",""talhaahad"",""aiseries"",""tcm""]",1656727755658052,False,en,"[""post_text""]",https://www.facebook.com/transparency-tools/content-library/dataset/1119037145491882/instagram/post/1656727755658052/,2026-05-13T13:13:23+00:00,"[{""id"":""1656727755658052"",""type"":""video"",""duration"":96.98,""tags"":{""0"":""815162807738463"",""1"":""363842936492074"",""2"":""1605108547564743"",""3"":""862811636124438""},""url"":""https://lookaside.facebook.com/mcl/multimedia/lookaside/?download_info=AZs-aB7bOGKbGjI-CcOOv4JNNXZrbNDrHSoeDEePp-cNP-QBiwXN0zt7na3Gm3UOJcS3xz0cxnZzMX8-t-DSIP3E0oVnPB_FEZTvYs01NDdf6bjNJc3IQyLO-kqRy4KnFzkuYs69f0sYV3EsVfmB""}]",...,0.0,7.0,644.0,2026-05-13,"Ask AI a wrong question, it'll give you a confident wrong answer.\nHow do you catch it?\nOnly if your fundamentals are solid.\n\nWatch the episode with @atom.camp on my youtube channel:\nThe link is in my Bio.\n\n#talhaahad #talhaahadroundtable #aiseries #tcm #talhaahadpodcast","['#talhaahad', '#talhaahadroundtable', '#aiseries', '#tcm', '#talhaahadpodcast']","Ask AI a wrong question, it'll give you a confident wrong answer. How do you catch it? Only if your fundamentals are solid. Watch the episode with @atom.camp on my youtube channel: The link is in my Bio.",2026-05-11,2025-07-21,False
1,videos,2026-05-13 10:49:04,"[""titancapitalmarkets"",""tcm""]",898235766608248,False,en,"[""post_text""]",https://www.facebook.com/transparency-tools/content-library/dataset/1119037145491882/instagram/post/898235766608248/,2026-05-13T11:39:08+00:00,"[{""id"":""898235766608248"",""type"":""video"",""duration"":42.26,""tags"":{""0"":""2541861929546394""},""url"":""https://lookaside.facebook.com/mcl/multimedia/lookaside/?download_info=AZuPZBr281UWDYmMxBFdpF7q_HRsA7IHXJayOwult3szEQfL87-Tj6PeXpAveoOKLNL-CC5nYmIq46JkqheKVGD7rAZ9-XTBvX3hbNROLYptzADWElKWRlpUO94g524jJhPa3qsKROJuBuA6Ue4""}]",...,3.0,37.0,1339.0,2026-05-13,"From making hits…\nto hitting payouts.\n\n5ivebeatz took the same discipline, consistency, and focus from music and applied it to trading.\n\nNow it’s paying at scale.\n\n#titancapitalmarkets #tcm","['#titancapitalmarkets', '#tcm']","From making hits… to hitting payouts. 5ivebeatz took the same discipline, consistency, and focus from music and applied it to trading. Now it’s paying at scale.",2026-05-11,2025-05-12,False
2,videos,2026-05-13 02:11:45,"[""healthy"",""missmaya"",""tcm"",""wellness"",""guasha""]",980906014454966,False,en,"[""post_text""]",https://www.facebook.com/transparency-tools/content-library/dataset/1119037145491882/instagram/post/980906014454966/,2026-05-13T02:42:14+00:00,"[{""id"":""980906014454966"",""type"":""video"",""duration"":45.3,""url"":""https://lookaside.facebook.com/mcl/multimedia/lookaside/?download_info=AZtbkpMMe80Yo02bIqwpLH_xfjNewRnxw3W-S-04QtJFjR_CTyv7EMiLD0eWDy71hZOqRIlAJxw-clZEadlcjfOqmFs852OTRG25zEOj-_I1MrriR0oXe-33Da0WkeUZOyARoveGh0DTVl0l_48""}]",...,0.0,3.0,861.0,2026-05-13,#tcm #missmaya #guasha #healthy #wellness,"['#tcm', '#missmaya', '#guasha', '#healthy', '#wellness']",NaN,2026-05-11,2025-09-01,False
3,albums,2026-05-13 01:57:36,"[""chinesemedicineworks"",""pmos"",""tcm"",""acupuncture"",""pcos""]",1016501597388882,False,en,"[""post_text""]",https://www.facebook.com/transparency-tools/content-library/dataset/1119037145491882/instagram/post/1016501597388882/,2026-05-13T01:59:21+00:00,"[{""id"":""744512645356645"",""type"":""photo"",""url"":""https://lookaside.facebook.com/mcl/multimedia/lookaside/?download_info=AZuTYeNjVDiGNxQ5mKY-ZdtBJF_TINkNTuq5cU0AU2KwTcoRukWudD7WmrkG0mTt8irenEitKfvF44FaOL9elWGPneOiQM_8FmCuDN6FwldYujkKlf-sd3LHs-_KXOmZRCP5M3naf4ntE27GPOQ""},{""id"":""2141712803273281